In [3]:
import os, sys

project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

print("Project root:", project_root)
print("Src path:", src_path)


Project root: c:\Users\test\Documents\bias_lense
Src path: c:\Users\test\Documents\bias_lense\src


In [4]:
from datasets import load_dataset

dataset = load_dataset("AyoubChLin/CNN_News_Articles_2011-2022", split="train")

print(dataset)
print("Example row:")
print(dataset[0])


c:\Users\test\Documents\bias_lense\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'label'],
    num_rows: 32218
})
Example row:
{'text': ' (CNN)Celebrities such as Piers Morgan and nearly 800 other members of a London golf club will earn £85,000 ($107,100) each after agreeing the sale of their Wimbledon Park Golf Club to the organizer of the Wimbledon grand slam tennis tournament.The All England Lawn Tennis Club (AELTC) has acquired the land, which lies across the road from the tournament venue, for £65 million ($81.9 million) with a long-term view to transferring Wimbledon\'s qualifying event to the 73-acre site and building over the golf greens.The two parties have been in discussion for a decade but members of the 120-year-old golf club had resisted a number of lower bids before 82% agreed to Wimbledon\'s "best and final offer" at a meeting in central London Thursday. Very sad news.Played there for over 30yrs & voted against the sale. Hope the superb pro-shop team get properly looked after. https://t.co/Et4ATm5eMl— Piers Morgan (@

In [9]:
TEXT_COL = "text"
CATEGORY_COL = "label"   
SAMPLE_SIZE = 300 

ds_small = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE))

docs = []
for i, row in enumerate(ds_small):
    raw_text = row.get(TEXT_COL, "")
    raw_cat = row.get(CATEGORY_COL, "Unknown")

    if raw_text is None:
        continue

    text = str(raw_text).strip()
    if len(text) < 50:
        continue

    words = text.split()
    if len(words) > 300:
        text = " ".join(words[:300])

    category = str(raw_cat)

    docs.append(
        {
            "id": len(docs),
            "title": f"[{category}] {text[:80].replace('\n', ' ')}...",
            "text": text,
        }
    )

len(docs), docs[0]


(300,
 {'id': 0,
  'title': '[3] Story highlightsA jewelry store in an upscale district of Paris was robbedAn est...',
  'text': "Story highlightsA jewelry store in an upscale district of Paris was robbedAn estimated 800,000 euros worth of luxury watches were stolenAnother jewelry store in the area was robbed in SeptemberOne of Paris' most upscale districts was once again the site of an armed jewelry store robbery Thursday, police said. Police said a number of luxury watches were stolen in a midday heist near Place Vendome, just blocks from the presidential palace and other important buildings. A couple, each armed with a handgun, entered the store and managed to escape with several luxury watches, police said. Police declined to give an estimate for the value of the stolen goods, but CNN affiliate BFMTV reported that they were worth 800,000 euros (about U.S. $1.1 million). JUST WATCHED$1.3 million reward for Cannes jewelryReplayMore Videos ...MUST WATCH$1.3 million reward for Cannes j

In [10]:
from embeddings import build_index

index = build_index(docs)
print("Index built with", len(index["docs"]), "documents")


Index built with 300 documents


In [12]:
from embeddings import semantic_search
from summarizer import summarize
from classifier import analyze_emotion


def news_emotion_agent_cnn(query: str, index, k: int = 3):
    hits = semantic_search(index, query, k=k)
    enriched = []

    for h in hits:
        try:
            summary = summarize(h["text"])
            emotion = analyze_emotion(summary)
        except Exception as e:
            print("⚠️ Skipping one article due to error:", e)
            continue

        enriched.append(
            {
                "title": h["title"],
                "score": h["score"],
                "summary": summary,
                "emotion_label": emotion["label"],
                "emotion_scores": emotion["scores"],
            }
        )
    return enriched


In [13]:
queries = [
    "elections in the United States",
    "climate change and hurricanes",
    "stock market and inflation",
]

for q in queries:
    print("\n" + "#" * 80)
    print("QUERY:", q)
    print("#" * 80)

    results = news_emotion_agent_cnn(q, index, k=3)
    for r in results:
        print("\n" + "=" * 60)
        print("Title:", r["title"])
        print(f"Relevance score: {r['score']:.3f}")
        print("Summary:", r["summary"])
        print("Emotion:", r["emotion_label"], r["emotion_scores"])



################################################################################
QUERY: elections in the United States
################################################################################

Title: [4] (CNN)As Florida Gov. Ron DeSantis runs for president looks to the final year of ...
Relevance score: 0.443
Summary: Florida Gov. Ron DeSantis wants to create an Office of Election Crime and Security. The office will contain 52 staff members, including 20 sworn law enforcement officers and 25 non-sworn investigators.
Emotion: fear {'happiness': 0.00487637659534812, 'fear': 0.04664449393749237, 'motivation': 0.0}

Title: [4] Washington (CNN)In some corners of the political internet, there are still some ...
Relevance score: 0.417
Summary: Some are still making the case that the 2018 election was not, in fact, a Democratic wave. But the facts make plain that 2018 was a massive and historic one. A 38-seat loss is the third-largest change of seats in the post-Watergate era.
Emotion

In [ ]:
emotional_docs = [
    {
        "id": 1,
        "title": "This policy is a complete disaster for normal people",
        "text": (
            "I am absolutely furious about the new government policy. "
            "It feels like they don't care about ordinary people at all. "
            "Everything is getting more expensive and we are the ones paying the price. "
            "This decision is unfair, selfish and honestly terrifying for our future."
        ),
    },
    {
        "id": 2,
        "title": "This breakthrough gives me real hope for the future",
        "text": (
            "This is honestly one of the most inspiring pieces of news I've read this year. "
            "It gives me real hope for the future. The people behind this project are brilliant "
            "and their work could change millions of lives for the better. "
            "I feel excited, grateful and motivated to contribute in any way I can."
        ),
    },
    {
        "id": 3,
        "title": "I’m scared about where the world is heading",
        "text": (
            "Every day I wake up and read the news and I feel more scared. "
            "Wars, crises, corruption everywhere. It feels like nobody is in control. "
            "I’m worried about my family, my friends and my own future. "
            "It's hard not to feel anxious and hopeless when things look this dark."
        ),
    },
    {
        "id": 4,
        "title": "This project motivates me to work harder than ever",
        "text": (
            "Working on this project has been the most motivating experience in my life. "
            "Even when I’m tired, I feel a huge drive to keep going, learn more and build something meaningful. "
            "I know the road will be difficult, but I’m determined to push through every obstacle and grow."
        ),
    },
    {
        "id": 5,
        "title": "I feel pure joy watching this community come together",
        "text": (
            "Seeing this community come together fills me with pure joy. "
            "People are helping each other, sharing resources, and celebrating every small success. "
            "It's heartwarming and uplifting, and it reminds me that there is still so much kindness in the world."
        ),
    },
]
len(emotional_docs)


5

In [15]:
from embeddings import build_index

emotional_index = build_index(emotional_docs)
print("Emotional index size:", len(emotional_index["docs"]))


Emotional index size: 5


In [16]:
from embeddings import semantic_search
from classifier import analyze_emotion


def news_emotion_agent_emotional(query: str, index=emotional_index, k: int = 3):
    """
    1) Semantic search on emotional_docs
    2) Classify emotion of the FULL text (no summarization)
    """
    hits = semantic_search(index, query, k=k)
    enriched = []

    for h in hits:
        emotion = analyze_emotion(h["text"])

        enriched.append(
            {
                "title": h["title"],
                "score": h["score"],
                "text": h["text"],
                "emotion_label": emotion["label"],
                "emotion_scores": emotion["scores"],
            }
        )
    return enriched


In [17]:
queries = [
    "unfair government policy",
    "hope for the future",
    "I feel scared about the world",
    "this project motivates me",
]

for q in queries:
    print("\n" + "#" * 80)
    print("EMOTIONAL QUERY:", q)
    print("#" * 80)

    results = news_emotion_agent_emotional(q, emotional_index, k=2)
    for r in results:
        print("\n" + "=" * 60)
        print("Title:", r["title"])
        print(f"Relevance score: {r['score']:.3f}")
        print("Text (shortened):", r["text"][:200] + "...")
        print("Emotion:", r["emotion_label"], r["emotion_scores"])



################################################################################
EMOTIONAL QUERY: unfair government policy
################################################################################

Title: This policy is a complete disaster for normal people
Relevance score: 0.509
Text (shortened): I am absolutely furious about the new government policy. It feels like they don't care about ordinary people at all. Everything is getting more expensive and we are the ones paying the price. This dec...
Emotion: fear {'happiness': 0.0013668795581907034, 'fear': 0.4692606031894684, 'motivation': 0.0}

Title: This breakthrough gives me real hope for the future
Relevance score: 0.081
Text (shortened): This is honestly one of the most inspiring pieces of news I've read this year. It gives me real hope for the future. The people behind this project are brilliant and their work could change millions o...
Emotion: happiness {'happiness': 0.9810913801193237, 'fear': 0.0005985477473586798, 'm